<a href="https://colab.research.google.com/github/Ali-mohammadi-design/6220-Project-winter-2022/blob/main/CHAT_With_Documents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pymupdf faiss-cpu sentence-transformers

# 📁 Upload PDFs
from google.colab import files
import os

uploaded = files.upload()
os.makedirs("pdfs", exist_ok=True)
for name in uploaded:
    os.rename(name, f"pdfs/{name}")
print("✅ PDFs uploaded.")

Saving Sebastian Raschka - Build a Large Language Model (From Scratch)-Manning (2024).pdf to Sebastian Raschka - Build a Large Language Model (From Scratch)-Manning (2024).pdf
✅ PDFs uploaded.


In [2]:
# 🧠 PDF Chat Engine
import os
import fitz  # PyMuPDF
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from typing import List, Tuple

model = SentenceTransformer('all-MiniLM-L6-v2')


class PDFChatEngine:
    def __init__(self):
        self.chunks: List[str] = []
        self.chunk_sources: List[Tuple[str, int]] = []
        self.index = None

    def load_pdfs(self, folder_path: str):
        for filename in os.listdir(folder_path):
            if filename.lower().endswith(".pdf"):
                self._parse_pdf(os.path.join(folder_path, filename), filename)
        self._build_index()

    def _parse_pdf(self, path: str, name: str):
        doc = fitz.open(path)
        for i, page in enumerate(doc):
            text = page.get_text()
            if text.strip():
                self.chunks.append(text)
                self.chunk_sources.append((name, i + 1))
        doc.close()

    def _build_index(self):
        embeddings = model.encode(self.chunks)
        dimension = embeddings[0].shape[0]
        self.index = faiss.IndexFlatL2(dimension)
        self.index.add(np.array(embeddings))

    def query(self, question: str, top_k: int = 5):
        question_embedding = model.encode([question])
        D, I = self.index.search(np.array(question_embedding), top_k)
        responses = []
        for idx in I[0]:
            src = self.chunk_sources[idx]
            text = self.chunks[idx]
            responses.append((src, text))
        return responses


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
# 🚀 Run Chat Interface
engine = PDFChatEngine()
engine.load_pdfs("pdfs")

print("✅ PDF content indexed. You can now chat below.")


✅ PDF content indexed. You can now chat below.


In [4]:
# 💬 Chat loop (text input)
while True:
    q = input("\nAsk a question (or 'exit'): ")
    if q.lower() in ["exit", "quit"]:
        break
    answers = engine.query(q)
    for (filename, page), content in answers:
        print(f"\n📄 [From {filename}, Page {page}]:\n{content[:500]}\n...")



Ask a question (or 'exit'): Please tell me about attention mechanism

📄 [From Sebastian Raschka - Build a Large Language Model (From Scratch)-Manning (2024).pdf, Page 73]:
51
parts of the LLM surrounding the self-attention mechanism to see it in action and to
create a model to generate text.
 We will implement four different variants of attention mechanisms, as illustrated in
figure 3.2. These different attention variants build on each other, and the goal is to
This chapter implements the
attention mechanism, an important
building block of GPT-like LLMs
1) Data
preparation
& sampling
2) Attention
mechanism
Building an LLM
STAGE 1
Foundation model
STAGE 2
STAGE 3
C
...

📄 [From Sebastian Raschka - Build a Large Language Model (From Scratch)-Manning (2024).pdf, Page 78]:
56
CHAPTER 3
Coding attention mechanisms
Since self-attention can appear complex, especially if you are encountering it for the
first time, we will begin by examining a simplified version of it. Then we will imple-
ment

KeyboardInterrupt: Interrupted by user